In [1]:
import re
import random
from collections import defaultdict, Counter

Normaliza el texto eliminando espacios extras y convirtiendo a minúsculas.
Como resultado texto limpio y en minúsculas.

In [2]:
def limpiar_texto(texto):
    # Quitar saltos de línea y caracteres extraños
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

Divide el texto en frases usando signos de puntuación.
Como resultado lista de frases.

In [3]:
def dividir_frases(texto):
    # Divide el texto en frases simples
    frases = re.split(r'(?<=[.!?]) +', texto)
    return [f.strip() for f in frases if f.strip()]

Construye modelos de unigramas, bigramas y trigramas a partir del texto.
La entrada es texto limpio (ya tokenizado por espacios).
Retorna tuple: (unigrama, bigrama, trigrama) donde cada uno es un contador de frecuencias.

In [4]:
def entrenar_modelo(texto):
    # Tokenización simple por espacios
    palabras = texto.split()
    # Contadores de n-gramas
    unigrama = Counter(palabras)
    bigrama = defaultdict(Counter)
    trigrama = defaultdict(Counter)

    # Construcción de bigramas
    for i in range(len(palabras)-1):
        bigrama[(palabras[i],)][palabras[i+1]] += 1

    # Construcción de trigramas
    for i in range(len(palabras)-2):
        trigrama[(palabras[i], palabras[i+1])][palabras[i+2]] += 1

    return unigrama, bigrama, trigrama

Predice la siguiente palabra usando interpolación backoff:
1) Trigrama si hay contexto de 2 palabras.
2) Bigrama si hay contexto de 1 palabra.
3) Unigrama en caso contrario.

In [5]:
def predecir_siguiente(estado, unigrama, bigrama, trigrama):
    # Caso trigramas
    if len(estado) >= 2:
        clave_tri = tuple(estado[-2:])
        if clave_tri in trigrama and trigrama[clave_tri]:
            opciones = list(trigrama[clave_tri].keys())
            pesos = list(trigrama[clave_tri].values())
            return random.choices(opciones, weights=pesos)[0]

    # Caso bigramas
    if len(estado) >= 1:
        clave_bi = (estado[-1],)
        if clave_bi in bigrama and bigrama[clave_bi]:
            opciones = list(bigrama[clave_bi].keys())
            pesos = list(bigrama[clave_bi].values())
            return random.choices(opciones, weights=pesos)[0]

    # Caso unigramas (fallback)
    opciones = list(unigrama.keys())
    pesos = list(unigrama.values())
    return random.choices(opciones, weights=pesos)[0]

In [6]:
def generar_texto(unigrama, bigrama, trigrama, longitud=20):
    inicio = random.choice(list(unigrama.keys()))
    estado = [inicio]
    salida = [inicio]

    for _ in range(longitud-1):
        siguiente = predecir_siguiente(tuple(estado), unigrama, bigrama, trigrama)
        salida.append(siguiente)
        estado.append(siguiente)
    return " ".join(salida)


Extrae las palabras más frecuentes (sin stopwords) como palabras clave.

In [7]:
def extraer_palabras_clave(texto, top_k=10):
   palabras = re.findall(r'\w+', texto.lower())
   stopwords = {"de", "la", "y", "el", "en", "los", "las", "un", "una", 
                 "por", "para", "que", "del", "se", "con", "su", "es"}
   filtradas = [p for p in palabras if p not in stopwords and len(p) > 2]
   return [w for w, _ in Counter(filtradas).most_common(top_k)]

Asigna un puntaje a cada frase basado en:
- Frecuencia de sus bigramas en el corpus.
- Cantidad de palabras clave contenidas.

Parámetros:
    frases (list): Lista de frases del texto original.
    conteos (tuple): (unigrama, bigrama, trigrama).
    palabras_clave (list): Palabras clave extraídas.

Retorna:
    dict: Diccionario frase -> puntaje.

In [8]:
def puntuar_frases(frases, conteos, palabras_clave):
    # Asigna un puntaje a cada frase basado en:
    puntuaciones = {}
    for frase in frases:
        palabras = frase.lower().split()
        score = 0
        count = 0
        for i in range(len(palabras)-1):
            par = (palabras[i],)
            if par in conteos[1]:  # bigrama
                score += sum(conteos[1][par].values())
                count += 1
        score += sum(2 for p in palabras if p in palabras_clave)
        puntuaciones[frase] = score / max(1, count)
    return puntuaciones

Genera un resumen extractivo con suavizado:
- Selecciona frases importantes.
- Si son muy cortas, las extiende con el modelo interpolado.

Parámetros:
    texto (str): Texto original.
    num_frases (int): Número de frases a incluir en el resumen.

Retorna:
    tuple: (resumen, palabras_clave)

In [9]:
def resumen_markov_backoff(texto, num_frases=3):
    """
    Genera un resumen extractivo con suavizado:
    - Selecciona frases importantes.
    - Si son muy cortas, las extiende con el modelo interpolado.

    Parámetros:
        texto (str): Texto original.
        num_frases (int): Número de frases a incluir en el resumen.

    Retorna:
        tuple: (resumen, palabras_clave)
    """
    texto_limpio = limpiar_texto(texto)
    frases = dividir_frases(texto)
    unigrama, bigrama, trigrama = entrenar_modelo(texto_limpio)
    palabras_clave = extraer_palabras_clave(texto_limpio, top_k=10)

    conteos = (unigrama, bigrama, trigrama)
    puntuaciones = puntuar_frases(frases, conteos, palabras_clave)

    # Seleccionar frases más relevantes
    frases_importantes = sorted(puntuaciones, key=puntuaciones.get, reverse=True)[:num_frases]

    # Suavizado: extender frases cortas
    resumen_suavizado = []
    for f in frases_importantes:
        palabras = f.split()
        if len(palabras) < 8:
            extendida = generar_texto(unigrama, bigrama, trigrama, inicio=palabras[-1], longitud=10)
            resumen_suavizado.append(f + " " + extendida)
        else:
            resumen_suavizado.append(f)

    return " ".join(resumen_suavizado), palabras_clave

In [13]:
texto = """
Retomar la Sesión Anterior y Sinergia con COE Juan Felipe Gomez Velez reanudó la reunión, retomando la sesión anterior sobre lineamientos técnicos e invitó a Luz Andrea García Trujillo para que aportara su contexto (00:03:59). El objetivo de estos encuentros es lograr la sinergia entre arquitectos transversales y el COE en cuanto a lineamientos, modelo de gobierno, modelo operativo y artefactos, para trabajar de manera unificada. Se busca entender cómo están trabajando los arquitectos para integrar sus buenas prácticas en una metodología común, en lugar de imponer procesos (00:06:12).
Visión del COE y Sofka Jhonatan Brian del Chiaro Collante planteó la necesidad de alinear la visión de Sofka y el COE con la visión de los arquitectos, señalando que el objetivo del COE es "impulsar la excelencia técnica y fomentar la innovación a través del trabajo colaborativo" (00:08:53) (00:10:38). Los participantes acordaron que esta visión del COE sigue siendo relevante por el momento, aunque se requerirá validación con las capas estratégica y ejecutiva de la compañía. Además, se buscó la visión de Sofka como empresa para alinearla con la visión del COE y los arquitectos transversales, con la ayuda de Raúl Andrés Alzate Gómez para encontrarla en Gemini (00:11:32) (00:16:29).
Estado de los Hallazgos y Bloques de Construcción Jhonatan Brian del Chiaro Collante mencionó que se habían establecido hallazgos desde la primera reunión y que la idea era dar continuidad a esos hallazgos y a los bloques de construcción, como los lineamientos y principios (00:07:38). Propuso estructurar la visión de los arquitectos para luego definir principios y lineamientos (00:08:53). Juan Felipe Gomez Velez destacó que se ha avanzado en el modelo operativo y de gobierno, y que el enfoque es entregar valor que sea útil en el día a día de los arquitectos (00:09:46).
Uso del Tablero de Gestión Diego Henao Severiche sugirió revivir el tablero de gestión de tareas, que había sido discutido al inicio de estas reuniones, para que los arquitectos puedan avanzar en tareas relacionadas con artefactos o tiempos disponibles (00:12:46). Jhonatan Brian del Chiaro Collante compartió el tablero, el cual incluye secciones para tareas pendientes, herramientas de gestión, modelo de gobierno, gestión de servicio, hallazgos y un catálogo estandarizado de servicios (00:13:47).
Aclaración sobre Propósito y Visión Diego Henao Severiche preguntó si "visión" y "propósito" se referían a lo mismo. Jhonatan Brian del Chiaro Collante aclaró que el propósito es la misión actual, mientras que la visión es cómo se ven en el futuro (00:14:42). Se confirmó que la visión de Sofka es "transformar la vida de personas siendo arenalógicos más confiables", y la del COE sigue siendo "impulsar la excelencia técnica y fomentar la innovación a través del trabajo colaborativo" (00:17:42).
Seguimiento a Arquitectos Juan Felipe Gomez Velez indagó si en la Business Unit 2, la más madura, se realiza algún seguimiento a los arquitectos en cuanto a las herramientas que usan, cómo trabajan, su día a día y el cumplimiento de su rol (00:19:58). Mencionó el caso de un arquitecto en Honduras que no pudo responder una encuesta de caracterización por restricciones de acceso a sistemas de Sofka desde su máquina del cliente, lo que resalta la necesidad de mejorar la comunicación y el acceso (00:21:09).
Desafíos de Comunicación y Acceso en Clientes Carlos Alberto Coca Acosta afirmó que este problema de acceso se presenta en clientes como Diners y Pichincha, donde los equipos son muy restrictivos y las personas deben usar sus equipos personales, lo que dificulta la comunicación fluida (00:26:57). Juan Felipe Gomez Velez se comprometió a llevar el tema a Gloria y Marta para mejorar la comunicación, buscando habilitar el dominio de Sofka para el correo sin infringir la seguridad (00:28:18).
Habilidades Blandas y Acompañamiento a Arquitectos Jhonatan Brian del Chiaro Collante propuso trabajar en las habilidades blandas de los arquitectos, como comunicación, negociación y resolución, sugiriendo una colaboración con el COE de management o People para actividades periódicas (00:24:24). Además, Raúl Andrés Alzate Gómez preguntó sobre la confianza en las evaluaciones de habilidades esenciales en el proceso de selección. Diego Henao Severiche comentó que a veces se contrata personal aunque no cumplan con todos los requisitos para cumplir con los tiempos de asignación, lo que puede afectar la calidad (00:51:39).
Procesos del COE y Desafíos del Onboarding Carlos Alberto Coca Acosta explicó que la transversal apoya en reclutamiento, preventas y bloqueantes, y que en el reclutamiento se identifican las brechas de los candidatos (00:38:48). Destacó la importancia de un buen onboarding que incluya el conocimiento de la estructura interna del cliente y de Sofka para que los arquitectos sepan con quién hablar y reciban apoyo (00:36:01) (00:40:41). Se identificaron varios procesos clave como onboarding, preventa, cierre de brechas y revisión de arquitectura (00:57:18).
Normalización de Roles y Habilidades Luz Andrea García Trujillo mencionó que a menudo los profesionales son promovidos sin una validación formal de aptitudes para el nuevo rol, lo que lleva a la necesidad de una visión unificada de lo que debería ser un líder técnico (00:46:29) (00:48:49). Carlos Alberto Coca Acosta corroboró que en su Business Unit se tiene en cuenta al COE para la validación de perfiles, pero que hay casos donde los clientes piden roles híbridos que no encajan en las definiciones existentes, dejando a las personas sin un claro soporte (00:47:40) (00:50:07).
Estandarización de Servicios y Próximos Pasos Jhonatan Brian del Chiaro Collante propuso estandarizar los servicios de los arquitectos transversales para crear un catálogo (00:44:20). Carlos Alberto Coca Acosta compartió que han mapeado el proceso de preventa y de cierre de brechas, sugiriendo que se analicen la próxima semana para identificar similitudes y diferencias entre las Business Units y lograr un enfoque unificado (00:56:10).
Temas Operacionales y de Gobierno Jhonatan Brian del Chiaro Collante y Carlos Alberto Coca Acosta discutieron la necesidad de apoyo operacional y la supervisión del gobierno. Jhonatan Brian del Chiaro Collante sugirió que el gobierno debería supervisar varios aspectos, incluyendo la efectividad del apoyo operacional y el servicio de preventa. Carlos Alberto Coca Acosta también destacó la falta de retroalimentación de las preventas a la operación, lo que resulta en que dediquen tiempo a la operación para asegurar una transición fluida (00:59:28).
"""


resumen, keywords = resumen_markov_backoff(texto, num_frases=3)

print("🔹 Palabras clave detectadas:\n", keywords)
print("\n🔹 Resumen con suavizado e interpolación:\n", resumen)

🔹 Palabras clave detectadas:
 ['arquitectos', 'visión', 'coe', 'jhonatan', 'brian', 'chiaro', 'collante', 'sofka', 'gobierno', 'carlos']

🔹 Resumen con suavizado e interpolación:
 Carlos Alberto Coca Acosta compartió que han mapeado el proceso de preventa y de cierre de brechas, sugiriendo que se analicen la próxima semana para identificar similitudes y diferencias entre las Business Units y lograr un enfoque unificado (00:56:10).
Temas Operacionales y de Gobierno Jhonatan Brian del Chiaro Collante y Carlos Alberto Coca Acosta discutieron la necesidad de apoyo operacional y la supervisión del gobierno. Además, Raúl Andrés Alzate Gómez preguntó sobre la confianza en las evaluaciones de habilidades esenciales en el proceso de selección. Además, se buscó la visión de Sofka como empresa para alinearla con la visión del COE y los arquitectos transversales, con la ayuda de Raúl Andrés Alzate Gómez para encontrarla en Gemini (00:11:32) (00:16:29).
Estado de los Hallazgos y Bloques de Construc